# Build the production 2025 model-feature store

This notebook is the auditable orchestration record for the complete feature pipeline. The production run has achieved the original objective: it created validated, resumable feature-family stores and the matched root/all-comment choice sets consumed by 06B and 06C. Long transformer jobs are launched as shell commands on the Linux A5000 host; ordinary notebook execution inspects their manifests and validates the completed outputs without recomputing them.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pyarrow.parquet as pq

DATA_ROOT = Path(os.getenv("COMMENTGAP_DATA_ROOT", "data/scrape_2025"))
MODEL_ROOT = Path(os.getenv("COMMENTGAP_MODEL_ROOT", "model_output/selection_2025"))
EMBEDDING_ROOT = MODEL_ROOT / "embeddings"
SIMILARITY_ROOT = MODEL_ROOT / "similarities"
AQUA_ROOT = MODEL_ROOT / "aqua"
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", str(MODEL_ROOT / "features")))

qa = json.loads((DATA_ROOT / "qa_summary/year=2025/summary.json").read_text())
assert qa["passed"] and qa["nonterminal_stories"] == 0
{
    "articles": qa.get("article_rows"),
    "comments": qa.get("comment_rows"),
    "status_counts": qa.get("status_counts"),
    "sticky_descendant_rows": qa.get("sticky_descendant_rows"),
}

## Production dependency graph

1. **BGE-M3 embeddings:** normalized comment and article-passage vectors for December 2024 and 2025.
2. **Semantic scalars:** article similarity plus strict-prior all-comment and root-comment novelty for 2025.
3. **AQuA:** twenty German deliberative-quality dimensions in the isolated Python 3.10 runtime; their continuous expected ordinal scores enter the primary models. Hard labels and composites remain descriptive/sensitivity features.
4. **Main feature assembly:** discussion and author history, deterministic local-text measures, CardiffNLP XLM-T sentiment, TextDetox toxicity, semantic joins, AQuA joins, length adjustments, and matched root/all choice sets.

Each expensive family has its own checkpoint identity. Repeating a compatible command resumes or reuses completed story shards; do not add `--overwrite` to a routine restart.

## 1. BGE-M3 embedding store — long CUDA run

The production run omitted `--year`, deliberately allowing discovery of both December 2024 and all 2025 partitions. It began at batch 64 and adaptively increased to 256; vectors were stored as float16 while inference remained float32.

```bash
source .venv/bin/activate
python -m pip install -e .

commentgap-embed \
  --model-id BAAI/bge-m3 \
  --revision 5617a9f61b028005a4858fdac845db406aefb181 \
  --device cuda \
  --batch-size 64 \
  --max-batch-size 256 \
  --min-batch-size 4 \
  --storage-dtype float16 \
  --progress-every-stories 25
```

The completed manifest recorded 48,061 articles, 9,941,495 eligible comments, a 1,024-dimensional embedding, and the `COMPLETE_SOURCE` watermark.

## 2. Precomputed semantic features — long CPU/RAM run

This stage reads the completed embedding store and does not reload BGE-M3. The 2025 model requires article similarity, novelty from earlier comments, and novelty from earlier roots.

```bash
commentgap-similarity \
  --model-id BAAI/bge-m3 \
  --revision 5617a9f61b028005a4858fdac845db406aefb181 \
  --year 2025 \
  --exact-novelty-threshold 5000 \
  --progress-every-stories 100
```

The production similarity build signature is resolved from its manifest; the feature assembler rejects a source-fingerprint mismatch.

## 3. AQuA deliberative-quality store — long isolated CUDA run

The numerical production source was run with the following tuned A5000 settings: batch ceiling 128, padded-token budget 2,048, and cross-story windows of at most 500 stories or 200,000 rows. The historical run accidentally included `--allow-unverified-parity`, so it was correctly watermarked as a pilot and subsequently promoted after upstream parity verification. **For a new production run with verified parity, omit that flag.**

```bash
# Historical numerical run (retained here for exact reproducibility).
commentgap-aqua \
  --runtime-python .venv-aqua/bin/python \
  --device cuda \
  --batch-size 128 \
  --max-batch-tokens 2048 \
  --window-max-stories 500 \
  --window-max-rows 200000 \
  --progress-every-stories 100 \
  --allow-unverified-parity

# Audited metadata promotion after commentgap-aqua-parity verified the fixture.
commentgap-aqua-promote \
  --source-store model_output/selection_2025/aqua \
  --output-root model_output/selection_2025/aqua \
  --data-root data/scrape_2025 \
  --year 2025 \
  --artifact-manifest aqua_runtime/artifacts.json \
  --progress-every-stories 25
```

Parity-fixture preparation and verification are documented in `notes/AQUA_PILOT_PROMOTION.md`. Promotion validated all 9,222,852 rows across 44,022 stories and changed only signatures/watermarks; numeric predictions were not recomputed.

## 4. Main feature assembly — long CUDA plus CPU/RAM run

This is the stage that fulfills the original purpose of 06A. It resolves and validates the similarity and production AQuA stores; reuses or creates local-text, CardiffNLP sentiment, and TextDetox toxicity story shards; computes strict-prior discussion/author history with the December 2024 lookback; and writes the final choice sets. The default immutable revisions are CardiffNLP `f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8` and TextDetox `cde9d07ac4df2af9c02d2461dee068bf04a58728`.

Production command:

```bash
commentgap-features \
  --data-root data/scrape_2025 \
  --similarity-root model_output/selection_2025/similarities \
  --aqua-store model_output/selection_2025/aqua \
  --lookback-root data/scrape_2025 \
  --year 2025 \
  --device cuda \
  --sentiment-batch-size 32 \
  --toxicity-batch-size 16 \
  --progress-every-stories 100
```

`--lookback-root data/scrape_2025` points at the same multi-year collection root and supplies its `year=2024` partition. Do not use `--overwrite` when resuming.

In [ ]:
%%bash
set -euo pipefail

if [[ "${COMMENTGAP_RUN_LONG_FEATURE_ASSEMBLY:-0}" != "1" ]]; then
  echo "Inspection mode: set COMMENTGAP_RUN_LONG_FEATURE_ASSEMBLY=1 before starting Jupyter to execute the production assembly."
  exit 0
fi

commentgap-features \
  --data-root data/scrape_2025 \
  --similarity-root model_output/selection_2025/similarities \
  --aqua-store model_output/selection_2025/aqua \
  --lookback-root data/scrape_2025 \
  --year 2025 \
  --device cuda \
  --sentiment-batch-size 32 \
  --toxicity-batch-size 16 \
  --progress-every-stories 100

## Acceptance checks

These checks are deliberately manifest- and Parquet-metadata-based, so they do not load the 214 MB root or 721 MB all-comment choice set into pandas. The deeper DuckDB key/null/selection audit was run separately and passed.

In [ ]:
state = json.loads((FEATURE_ROOT / "build_state.json").read_text())
provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
registry = json.loads((FEATURE_ROOT / "feature_manifest.json").read_text())

assert state["status"] == "complete"
assert provenance["watermark"] == "INFERENCE"
assert provenance["models"]["aqua"]["watermark"] == "PRODUCTION"
assert provenance["execution"]["choice_assembly_strategy"] == "bounded_cross_story_windows"

expected = [
    FEATURE_ROOT / "choice_set_root.parquet",
    FEATURE_ROOT / "choice_set_all.parquet",
    FEATURE_ROOT / "tie_diagnostics_root.parquet",
    FEATURE_ROOT / "tie_diagnostics_all.parquet",
    FEATURE_ROOT / "feature_manifest.json",
    FEATURE_ROOT / "provenance_manifest.json",
    FEATURE_ROOT / "novelty_validation.json",
]
missing = [str(path) for path in expected if not path.exists()]
assert not missing, missing
assert not list(FEATURE_ROOT.rglob("*.tmp"))

for scope in ("root", "all"):
    rows = pq.ParquetFile(FEATURE_ROOT / f"choice_set_{scope}.parquet").metadata.num_rows
    tie_rows = pq.ParquetFile(FEATURE_ROOT / f"tie_diagnostics_{scope}.parquet").metadata.num_rows
    assert rows == provenance[scope]["candidate_rows"]
    assert tie_rows == provenance[scope]["eligible_stories"]
    model_features = registry["models"][scope]["features"]
    aqua_expected = [
        feature for feature in model_features
        if feature.startswith("aqua_") and feature.endswith("_expected")
    ]
    assert len(aqua_expected) == 20
    assert "aqua_score_expected" not in model_features

{
    "build_signature": state["build_signature"],
    "dataset_fingerprint": provenance["source"]["dataset_fingerprint"],
    "aqua_build_signature": provenance["models"]["aqua"]["build_signature"],
    "root": provenance["root"],
    "all": provenance["all"],
    "root_model_features": len(registry["models"]["root"]["features"]),
    "all_model_features": len(registry["models"]["all"]["features"]),
}

## Downstream handoff

`06B_stacked_selection_models.Rmd` and `06C_xgboost_rankers.ipynb` both read `choice_set_root.parquet` and `choice_set_all.parquet` and obtain the primary feature lists from `feature_manifest.json`. The AQuA-backed contract contains 40 root features and 44 all-comment features: the original CardiffNLP, TextDetox, semantic, temporal, history, and structural predictors plus all 20 continuous AQuA expected ordinal dimensions. The AQuA hard labels, both composites, and mean toxicity remain descriptive/sensitivity fields. After both model families are regenerated from this feature build, `06D_model_tables_plots.ipynb` assembles their publication outputs.